In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


In [2]:
# Build a .py script that takes a snapshot date, loads a model artefact and make an inference and save to datamart

## set up pyspark session

In [3]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/17 13:23:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [4]:
snapshot_date_str = "2023-01-01"
model_name = "xgboostv1.pkl"


In [5]:
config = {}
config["snapshot_date_str"] = snapshot_date_str
config["snapshot_date"] = datetime.strptime(config["snapshot_date_str"], "%Y-%m-%d")
config["model_name"] = model_name
config["model_bank_directory"] = "model_bank/"
config["model_artefact_filepath"] = config["model_bank_directory"] + config["model_name"]

pprint.pprint(config)

{'model_artefact_filepath': 'model_bank/xgboostv1.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'xgboostv1.pkl',
 'snapshot_date': datetime.datetime(2023, 1, 1, 0, 0),
 'snapshot_date_str': '2023-01-01'}


## load model artefact from model bank

In [6]:
# Load the model from the pickle file
with open(config["model_artefact_filepath"], 'rb') as file:
    model_artefact = pickle.load(file)

print("Model loaded successfully! " + config["model_artefact_filepath"])

Model loaded successfully! model_bank/xgboostv1.pkl


## load feature store

In [7]:
import pandas as pd

merged_csv = "data/final_merged.csv"   # adjust path as needed

df = pd.read_csv(
    merged_csv,
    parse_dates=["snapshot_date"],
    low_memory=False
)

print(f"Loaded '{merged_csv}' with shape: {df.shape}")
print(df.head(10))   # show first 10 rows

Loaded 'data/final_merged.csv' with shape: (8974, 64)
                 loan_id Customer_ID  label   label_def snapshot_date   fe_1  \
0  CUS_0x1037_2023_01_01  CUS_0x1037      0  30dpd_6mob    2023-07-01   40.0   
1  CUS_0x1069_2023_01_01  CUS_0x1069      0  30dpd_6mob    2023-07-01  -26.0   
2  CUS_0x114a_2023_01_01  CUS_0x114a      0  30dpd_6mob    2023-07-01   35.0   
3  CUS_0x1184_2023_01_01  CUS_0x1184      0  30dpd_6mob    2023-07-01  278.0   
4  CUS_0x1297_2023_01_01  CUS_0x1297      1  30dpd_6mob    2023-07-01  162.0   
5  CUS_0x12fb_2023_01_01  CUS_0x12fb      0  30dpd_6mob    2023-07-01   56.0   
6  CUS_0x1325_2023_01_01  CUS_0x1325      0  30dpd_6mob    2023-07-01  -11.0   
7  CUS_0x1341_2023_01_01  CUS_0x1341      0  30dpd_6mob    2023-07-01  -68.0   
8  CUS_0x1375_2023_01_01  CUS_0x1375      1  30dpd_6mob    2023-07-01  -80.0   
9  CUS_0x13a8_2023_01_01  CUS_0x13a8      0  30dpd_6mob    2023-07-01  135.0   

    fe_2   fe_3   fe_4   fe_5  ...  Auto_Loan_count  \
0   90.0  

In [8]:
feature_location = "data/feature_clickstream.csv"

# Load CSV into DataFrame - connect to feature store
features_store_sdf = spark.read.csv(feature_location, header=True, inferSchema=True)
# print("row_count:",features_store_sdf.count())


# extract feature store
features_sdf = features_store_sdf.filter((col("snapshot_date") == config["snapshot_date"]))
print("extracted features_sdf", features_sdf.count(), config["snapshot_date"])

features_pdf = features_sdf.toPandas()
features_pdf

extracted features_sdf 8974 2023-01-01 00:00:00


,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,...,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20,Customer_ID,snapshot_date
0,63,118,80,121,55,193,111,112,-101,83,...,-16,-81,-126,114,35,85,-73,76,CUS_0x1037,2023-01-01
1,-108,182,123,4,-56,27,25,-6,284,222,...,-14,-96,200,35,130,94,111,75,CUS_0x1069,2023-01-01
2,-13,8,87,166,214,-98,215,152,129,139,...,26,86,171,125,-130,354,17,302,CUS_0x114a,2023-01-01
3,-85,45,200,89,128,54,76,51,61,139,...,172,96,174,163,37,207,180,118,CUS_0x1184,2023-01-01
4,55,120,226,-86,253,97,107,68,103,126,...,76,43,183,159,-26,104,118,184,CUS_0x1297,2023-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8969,242,54,14,-84,86,-6,29,22,-6,52,...,197,58,71,61,-141,243,133,77,CUS_0xdf6,2023-01-01
8970,155,200,71,-79,221,223,74,-10,124,211,...,-12,155,10,24,178,49,156,-153,CUS_0xe23,2023-01-01
8971,143,2,42,248,163,-13,-13,20,183,183,...,202,64,105,289,51,77,-20,89,CUS_0xe4e,2023-01-01
8972,109,161,161,29,57,122,-61,223,66,0,...,101,125,249,116,138,-8,160,94,CUS_0xedd,2023-01-01


## preprocess data for modeling and model prediction inference

In [9]:
import pandas as pd
import pickle
from sklearn.metrics import roc_auc_score, fbeta_score

# 1) Load the trained pipeline artifact
artifact_path = "model_bank/xgboostv1.pkl"
with open(artifact_path, "rb") as f:
    artefact = pickle.load(f)

pipe_final = artefact["model"]
best_thresh = 0.32
beta        = 2.0

# 2) Load your merged data
df = pd.read_csv(
    "data/final_merged.csv",
    parse_dates=["snapshot_date"],
    low_memory=False,
)

# 3) Pick the inference date
snapshot_date = pd.Timestamp("2024-09-01")
df_slice = df[df.snapshot_date == snapshot_date].reset_index(drop=True)

# 4) Define the raw feature columns (everything except your keys & label)
drop_cols = ["Customer_ID","snapshot_date","label_def","loan_id","Name","SSN","label"]
feature_cols = [c for c in df.columns if c not in drop_cols]

# 5) Extract X_raw and y_true (if you want to score)
X_raw = df_slice[feature_cols]
y_true = df_slice["label"]

# 6) Get predicted probabilities and binary preds
y_proba = pipe_final.predict_proba(X_raw)[:, 1]
y_pred  = (y_proba >= best_thresh).astype(int)

# 7) Compute your metrics
auc   = roc_auc_score(y_true, y_proba)
f2    = fbeta_score(y_true, y_pred, beta=beta)
gini  = 2*auc - 1

print(f"Inference on {snapshot_date.date()}:")
print(f" → Rows scored    : {len(X_raw)}")
print(f" → AUC            : {auc:.4f}")
print(f" → Gini           : {gini:.4f}")
print(f" → F{beta:.0f}-score : {f2:.4f}")

Inference on 2024-09-01:
 → Rows scored    : 511
 → AUC            : 0.7933
 → Gini           : 0.5866
 → F2-score : 0.7415


In [10]:
import pickle
import pandas as pd
from sklearn.metrics import roc_auc_score, fbeta_score, precision_score, recall_score

snapshot_date_str = "2024-09-01"
snapshot_date     = pd.Timestamp(snapshot_date_str)
model_path        = "model_bank/xgboostv1.pkl"
beta              = 2.0
threshold         = 0.32

with open(model_path, "rb") as f:
    artefact = pickle.load(f)

pipe_final = artefact["model"]

df = pd.read_csv("data/final_merged.csv",
                 parse_dates=["snapshot_date"],
                 low_memory=False)

df_slice = df[df.snapshot_date == snapshot_date].reset_index(drop=True)

features = list(pipe_final.feature_names_in_)
X = df_slice[features]
y = df_slice["label"]

proba = pipe_final.predict_proba(X)[:, 1]
preds = (proba >= threshold).astype(int)

auc   = roc_auc_score(y, proba)
gini  = 2 * auc - 1
f2    = fbeta_score(y, preds, beta=beta)
prec  = precision_score(y, preds)
rec   = recall_score(y, preds)

print(f"Metrics @ {snapshot_date_str}")
print(f"  F{beta:.0f}:       {f2:.4f}")
print(f"  AUC:       {auc:.4f}")
print(f"  Gini:      {gini:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")

out = df_slice[["Customer_ID", "snapshot_date"]].copy()
out["proba"]      = proba
out["prediction"] = preds

print("\nSample of individual outputs:")
print(out.head())

Metrics @ 2024-09-01
  F2:       0.7415
  AUC:       0.7933
  Gini:      0.5866
  Precision: 0.5147
  Recall:    0.8333

Sample of individual outputs:
  Customer_ID snapshot_date     proba  prediction
0  CUS_0x100b    2024-09-01  0.253899           0
1  CUS_0x1096    2024-09-01  0.273550           0
2  CUS_0x111c    2024-09-01  0.671239           1
3  CUS_0x112d    2024-09-01  0.816865           1
4  CUS_0x1204    2024-09-01  0.833386           1


## save model inference to datamart gold table

In [11]:
import os
import pickle
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.metrics import (
    roc_auc_score,
    fbeta_score,
    precision_score,
    recall_score
)
from pyspark.sql import SparkSession

In [12]:
from pyspark.sql import SparkSession
import os

# 1) Create SparkSession (if you don’t already have one)
spark = SparkSession.builder.appName("WriteModelPredictions").getOrCreate()

# 2) Convert your pandas ‘out’ DataFrame to a Spark DataFrame
spark_df = spark.createDataFrame(out)

# 3) Define your gold path & ensure the directory exists
model_base     = "xgboostv1"                        # same as model_artefact["model_version"]
snapshot_date  = snapshot_date_str.replace("-", "_")  # e.g. "2024_09_01"
gold_directory = f"datamart/gold/model_predictions/{model_base}/"

os.makedirs(gold_directory, exist_ok=True)

# 4) Compose a filename (or partition) for this date
partition_name = f"{model_base}_predictions_{snapshot_date}.parquet"
output_path    = os.path.join(gold_directory, partition_name)

# 5) Write out as Parquet
spark_df.write.mode("overwrite").parquet(output_path)

print("Saved predictions to:", output_path)

Saved predictions to: datamart/gold/model_predictions/xgboostv1/xgboostv1_predictions_2024_09_01.parquet


In [13]:
# Read the file
pred_path = "datamart/gold/model_predictions/xgboostv1/xgboostv1_predictions_2024_09_01.parquet"
print("Loading predictions from:", pred_path)


pred_sdf = spark.read.parquet(pred_path)
pred_sdf.show(10, truncate=False)
print(f"Total rows: {pred_sdf.count()}")

Loading predictions from: datamart/gold/model_predictions/xgboostv1/xgboostv1_predictions_2024_09_01.parquet
+-----------+-------------------+------------------+----------+
|Customer_ID|snapshot_date      |proba             |prediction|
+-----------+-------------------+------------------+----------+
|CUS_0xb646 |2024-09-01 00:00:00|0.6418221592903137|1         |
|CUS_0xb66f |2024-09-01 00:00:00|0.8448898792266846|1         |
|CUS_0xb676 |2024-09-01 00:00:00|0.2849210202693939|0         |
|CUS_0xb681 |2024-09-01 00:00:00|0.8354640603065491|1         |
|CUS_0xb69a |2024-09-01 00:00:00|0.205488383769989 |0         |
|CUS_0xb6d0 |2024-09-01 00:00:00|0.2792002856731415|0         |
|CUS_0xb6e7 |2024-09-01 00:00:00|0.6879045367240906|1         |
|CUS_0xb794 |2024-09-01 00:00:00|0.3104482889175415|0         |
|CUS_0xb837 |2024-09-01 00:00:00|0.8302323818206787|1         |
|CUS_0xb8c  |2024-09-01 00:00:00|0.8091781735420227|1         |
+-----------+-------------------+------------------+-------

## backfill

In [16]:
# set up config
snapshot_date_str = "2023-07-01"

start_date_str = "2023-07-01"
end_date_str = "2024-12-01"

In [17]:
# backfill.py

import os
from datetime import datetime
from model_inference import main as run_inference

def generate_first_of_month_dates(start_date_str, end_date_str):
    start = datetime.strptime(start_date_str, "%Y-%m-%d")
    end   = datetime.strptime(end_date_str,   "%Y-%m-%d")

    current = datetime(start.year, start.month, 1)
    out = []
    while current <= end:
        out.append(current.strftime("%Y-%m-%d"))
        # advance one month
        if current.month == 12:
            current = datetime(current.year + 1, 1, 1)
        else:
            current = datetime(current.year, current.month + 1, 1)
    return out

if __name__ == "__main__":
    MODEL_NAME = "xgboostv1.pkl"
    # backfill from Jan 2023 through Dec 2024:
    dates = generate_first_of_month_dates("2023-01-01", "2024-12-01")

    for dt in dates:
        print(f"\n>>> Scoring {dt} with {MODEL_NAME}")
        run_inference(dt, MODEL_NAME)


>>> Scoring 2023-01-01 with xgboostv1.pkl

--- STARTING inference for 2023-01-01 ---

{'artifact_path': 'model_bank/xgboostv1.pkl',
 'model_bank_directory': 'model_bank',
 'model_name': 'xgboostv1.pkl',
 'snapshot_date': Timestamp('2023-01-01 00:00:00'),
 'snapshot_date_str': '2023-01-01'}
Loaded artifact: model_bank/xgboostv1.pkl
Rows on 2023-01-01 → 0
→ no data for this month, skipping.

>>> Scoring 2023-02-01 with xgboostv1.pkl

--- STARTING inference for 2023-02-01 ---

{'artifact_path': 'model_bank/xgboostv1.pkl',
 'model_bank_directory': 'model_bank',
 'model_name': 'xgboostv1.pkl',
 'snapshot_date': Timestamp('2023-02-01 00:00:00'),
 'snapshot_date_str': '2023-02-01'}
Loaded artifact: model_bank/xgboostv1.pkl
Rows on 2023-02-01 → 0
→ no data for this month, skipping.

>>> Scoring 2023-03-01 with xgboostv1.pkl

--- STARTING inference for 2023-03-01 ---

{'artifact_path': 'model_bank/xgboostv1.pkl',
 'model_bank_directory': 'model_bank',
 'model_name': 'xgboostv1.pkl',
 'snapshot

## Check datamart

In [20]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [25]:
import pandas as pd

path = "datamart/gold/model_predictions/xgboostv1/xgboostv1_predictions_2024_12_01.parquet"
df = pd.read_parquet(path)

print("Total rows:", len(df))
print(df.dtypes)
print(df.head(10))

Total rows: 498
Customer_ID              object
snapshot_date    datetime64[ns]
score                   float32
prediction                int64
dtype: object
  Customer_ID snapshot_date     score  prediction
0  CUS_0x10dd    2024-12-01  0.267686           0
1  CUS_0x1109    2024-12-01  0.765089           1
2  CUS_0x1286    2024-12-01  0.366512           1
3  CUS_0x12a8    2024-12-01  0.268219           0
4  CUS_0x1309    2024-12-01  0.291069           0
5  CUS_0x13f8    2024-12-01  0.256265           0
6  CUS_0x1472    2024-12-01  0.305928           0
7  CUS_0x15a3    2024-12-01  0.277014           0
8  CUS_0x15f4    2024-12-01  0.163797           0
9  CUS_0x15f8    2024-12-01  0.323417           1
